# Random Boxplot Dataset Generator
This notebook generates a comprehensive set of random boxplots (saved as PNG images) with varied configurations: vertical/horizontal orientation, varying numbers of groups and samples, with or without outliers/extrema, notches, means displayed, varied whisker settings, and labeled axes. It also includes functions to convert labels to Braille (not enabled by default) so you can toggle Braille labeling later.

In [17]:
# Imports
import os
import json
import uuid
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Ensure consistent style
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi':150})

In [18]:
# Braille helper functions (not used by default, but available)
def text_to_braille(text):
    braille_dict = {
        'a': '⠁', 'b': '⠃', 'c': '⠉', 'd': '⠙',
        'e': '⠑', 'f': '⠋', 'g': '⠛', 'h': '⠓',
        'i': '⠊', 'j': '⠚', 'k': '⠅', 'l': '⠇',
        'm': '⠍', 'n': '⠝', 'o': '⠕', 'p': '⠏',
        'q': '⠟', 'r': '⠗', 's': '⠎', 't': '⠞',
        'u': '⠥', 'v': '⠧', 'w': '⠺', 'x': '⠭',
        'y': '⠽', 'z': '⠵', ' ': ' ', '-': '⠤',
        ',': '⠠', '.': '⠨', '(': '⠷', ')': '⠾',
        ':': '⠱', '/': '⠌', '0': '⠚', '1': '⠁',
        '2': '⠃', '3': '⠉', '4': '⠙', '5': '⠑',
        '6': '⠋', '7': '⠛', '8': '⠓', '9': '⠊', '&': '⠯'
    }
    return ''.join(braille_dict.get(c, '?') for c in text.lower())

# number_to_braille similar to examples
def number_to_braille(number):
    digit_to_braille = {'0':'⠚','1':'⠁','2':'⠃','3':'⠉','4':'⠙','5':'⠑','6':'⠋','7':'⠛','8':'⠓','9':'⠊','-':'⠤',' .':'⠨'}
    s = str(number)
    out = ''
    for ch in s:
        out += digit_to_braille.get(ch, '?')
    return out

In [19]:
# Helper functions to generate varied random data and create boxplots

def generate_group_samples(n_groups, min_size=10, max_size=500,
                           distribution_choices=None,
                           inject_extrema=False,
                           extrema_prob=0.2):
    """Return a list of numpy arrays, one per group."""
    if distribution_choices is None:
        distribution_choices = ['normal', 'uniform', 'exponential', 'bimodal', 'skewed']
    groups = []
    for _ in range(n_groups):
        size = np.random.randint(min_size, max_size+1)
        dist = np.random.choice(distribution_choices)
        if dist == 'normal':
            data = np.random.normal(loc=np.random.uniform(-5,5),
                                    scale=np.random.uniform(0.5,3.0),
                                    size=size)
        elif dist == 'uniform':
            low = np.random.uniform(-10,0)
            high = low + np.random.uniform(1,20)
            data = np.random.uniform(low, high, size=size)
        elif dist == 'exponential':
            data = np.random.exponential(scale=np.random.uniform(0.5,5.0), size=size)
            if np.random.rand() < 0.5:
                data = data * np.random.choice([-1,1])
        elif dist == 'bimodal':
            a = np.random.normal(loc=np.random.uniform(-5,0), scale=np.random.uniform(0.2,2.0), size=size//2)
            b = np.random.normal(loc=np.random.uniform(0,5), scale=np.random.uniform(0.2,2.0), size=size - size//2)
            data = np.concatenate([a,b])
        elif dist == 'skewed':
            base = np.random.normal(size=size)
            data = np.sign(base) * (np.abs(base) ** np.random.uniform(1.2,3.0)) * np.random.uniform(0.5,3.0)
        else:
            data = np.random.normal(size=size)

        # Optionally inject extreme outliers
        if inject_extrema and np.random.rand() < extrema_prob:
            n_ext = np.random.randint(1, max(2, size//20))
            for _ in range(n_ext):
                if np.random.rand() < 0.5:
                    data = np.append(data, data.mean() + np.random.uniform(10,50))
                else:
                    data = np.append(data, data.mean() - np.random.uniform(10,50))

        # Shuffle a bit
        np.random.shuffle(data)
        groups.append(np.array(data))
    return groups


def create_and_save_boxplot(groups,
                            filename,
                            vertical=True,
                            showfliers=True,
                            notch=False,
                            showmeans=False,
                            whis='range',
                            widths=0.6,
                            xlabel='Category',
                            ylabel='Value',
                            title='Boxplot',
                            use_braille=False,
                            save_data=True):
    """Create a boxplot from a list of arrays and save PNG + metadata and optionally data.

    This implementation ensures whis_arg is always initialized and computes
    detailed statistics per group for annotation and metadata.
    """
    fig, ax = plt.subplots(figsize=(8,6))

    # Normalize whis into a value accepted by matplotlib.boxplot
    whis_arg = None
    if isinstance(whis, str) and whis.lower() == 'range':
        # Percentile bounds to use full data range
        whis_arg = [0, 100]
    elif isinstance(whis, (np.floating, np.integer)):
        whis_arg = float(whis)
    elif isinstance(whis, (float, int)):
        whis_arg = float(whis)
    elif isinstance(whis, (list, tuple, np.ndarray)):
        whis_arg = list(whis)
    else:
        # Fallback to 1.5*IQR if unknown
        whis_arg = 1.5

    # Helper to compute boxplot stats per group
    def compute_stats(arr, whis_val):
        a = np.asarray(arr)
        if a.size == 0:
            return None
        q1 = float(np.percentile(a, 25))
        median = float(np.percentile(a, 50))
        q3 = float(np.percentile(a, 75))
        iqr = float(q3 - q1)
        mean = float(np.mean(a))

        # Determine whisker endpoints according to whis_val
        if isinstance(whis_val, (list, tuple)) and len(whis_val) == 2:
            lw = float(np.percentile(a, whis_val[0]))
            uw = float(np.percentile(a, whis_val[1]))
        else:
            # treat whis_val as multiplier of IQR
            mv = float(whis_val)
            cutoff_low = q1 - mv * iqr
            cutoff_high = q3 + mv * iqr
            within_low = a[a >= cutoff_low]
            within_high = a[a <= cutoff_high]
            lw = float(within_low.min()) if within_low.size > 0 else float(a.min())
            uw = float(within_high.max()) if within_high.size > 0 else float(a.max())

        outliers = a[(a < lw) | (a > uw)]

        return {
            'min': float(a.min()),
            'q1': q1,
            'median': median,
            'q3': q3,
            'max': float(a.max()),
            'iqr': iqr,
            'whis_low': lw,
            'whis_high': uw,
            'mean': mean,
            'outliers': outliers.tolist()
        }

    stats_list = [compute_stats(g, whis_arg) for g in groups]

    # matplotlib.boxplot expects a sequence of arrays
    bp = ax.boxplot(groups, vert=vertical, patch_artist=True,
                    showfliers=showfliers, notch=notch, widths=widths,
                    whis=whis_arg, meanline=False, showmeans=showmeans)

    # Style boxes: white fill, black border; IQR, whiskers, caps, median, outliers all black
    for patch in bp.get('boxes', []):
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
        patch.set_linewidth(2)
    for whisker in bp.get('whiskers', []):
        whisker.set_color('black')
        whisker.set_linewidth(2)
    for cap in bp.get('caps', []):
        cap.set_color('black')
        cap.set_linewidth(2)
    for median_line in bp.get('medians', []):
        median_line.set_color('black')
        median_line.set_linewidth(2)
    # Means (if shown) as black markers
    if 'means' in bp:
        for m in bp['means']:
            try:
                m.set_marker('D')
                m.set_markerfacecolor('black')
                m.set_markeredgecolor('black')
                m.set_markersize(6)
            except Exception:
                pass
    # Fliers / outliers as black dots
    if 'fliers' in bp:
        for fl in bp['fliers']:
            fl.set_marker('o')
            fl.set_markerfacecolor('black')
            fl.set_markeredgecolor('black')
            fl.set_markersize(4)

    # Annotate key points for each group
    n_groups = len(groups)
    for idx, s in enumerate(stats_list):
        if s is None:
            continue
        pos = idx + 1
        if vertical:
            # draw small markers for extrema, median, mean (black to match plot)
            ax.scatter([pos], [s['min']], color='black', marker='v', s=18, zorder=6)
            ax.scatter([pos], [s['max']], color='black', marker='^', s=18, zorder=6)
            ax.scatter([pos], [s['median']], color='black', marker='s', s=20, zorder=7)
            ax.scatter([pos], [s['mean']], color='black', marker='D', s=20, zorder=7)
            # dashed guide lines from box to whisker endpoints
            ax.plot([pos, pos], [s['q1'], s['whis_low']], color='black', linestyle='--', linewidth=1, zorder=5)
            ax.plot([pos, pos], [s['q3'], s['whis_high']], color='black', linestyle='--', linewidth=1, zorder=5)
            # whisker endpoints
            ax.scatter([pos], [s['whis_low']], color='black', marker='o', s=16, zorder=6)
            ax.scatter([pos], [s['whis_high']], color='black', marker='o', s=16, zorder=6)
            # outliers
            if s['outliers']:
                ax.scatter([pos]*len(s['outliers']), s['outliers'], color='black', marker='o', s=18, zorder=8)
        else:
            ax.scatter([s['min']], [pos], color='black', marker='v', s=18, zorder=6)
            ax.scatter([s['max']], [pos], color='black', marker='^', s=18, zorder=6)
            ax.scatter([s['median']], [pos], color='black', marker='s', s=20, zorder=7)
            ax.scatter([s['mean']], [pos], color='black', marker='D', s=20, zorder=7)
            ax.plot([s['q1'], s['whis_low']], [pos, pos], color='black', linestyle='--', linewidth=1, zorder=5)
            ax.plot([s['q3'], s['whis_high']], [pos, pos], color='black', linestyle='--', linewidth=1, zorder=5)
            ax.scatter([s['whis_low']], [pos], color='black', marker='o', s=16, zorder=6)
            ax.scatter([s['whis_high']], [pos], color='black', marker='o', s=16, zorder=6)
            if s['outliers']:
                ax.scatter(s['outliers'], [pos]*len(s['outliers']), color='black', marker='o', s=18, zorder=8)

    # Autoscale then draw axis intersection at zero if within range
    ax.relim()
    ax.autoscale_view()
    if vertical:
        ymin, ymax = ax.get_ylim()
        if ymin <= 0.0 <= ymax:
            ax.axhline(0.0, color='gray', linewidth=1, linestyle=':')
    else:
        xmin, xmax = ax.get_xlim()
        if xmin <= 0.0 <= xmax:
            ax.axvline(0.0, color='gray', linewidth=1, linestyle=':')

    # Labels
    if use_braille:
        ax.set_title(text_to_braille(title))
        ax.set_xlabel(text_to_braille(xlabel))
        ax.set_ylabel(text_to_braille(ylabel))
    else:
        ax.set_title(title)
        ax.set_xlabel(xlabel if vertical else ylabel)
        ax.set_ylabel(ylabel if vertical else xlabel)

    # Tick labels for categories
    if vertical:
        ax.set_xticks(np.arange(1, n_groups+1))
        ax.set_xticklabels([f'G{i+1}' for i in range(n_groups)], rotation=45)
    else:
        ax.set_yticks(np.arange(1, n_groups+1))
        ax.set_yticklabels([f'G{i+1}' for i in range(n_groups)])

    plt.tight_layout()
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    plt.savefig(filename, dpi=300)
    plt.close(fig)

    # Save metadata and data
    meta = {
        'filename': filename,
        'vertical': bool(vertical),
        'n_groups': int(n_groups),
        'showfliers': bool(showfliers),
        'notch': bool(notch),
        'showmeans': bool(showmeans),
        'whis': str(whis),
        'widths': float(widths),
        'xlabel': xlabel,
        'ylabel': ylabel,
        'title': title,
        'generated_at': datetime.utcnow().isoformat() + 'Z',
        'stats_per_group': stats_list
    }
    meta_filename = filename.replace('.png', '.json')
    with open(meta_filename, 'w') as f:
        json.dump(meta, f, indent=2)

    if save_data:
        data_filename = filename.replace('.png', '.npz')
        np.savez_compressed(data_filename, *groups)

    return meta

In [20]:
# Main generation loop: create a configurable number of randomized boxplots

def generate_dataset(output_dir='boxplot_dataset', n_examples=200, seed=None):
    if seed is not None:
        np.random.seed(seed)

    os.makedirs(output_dir, exist_ok=True)
    manifest = []

    for i in range(n_examples):
        # Random configuration
        n_groups = np.random.randint(1, 8)  # 1 to 7 groups
        inject_extrema = np.random.rand() < 0.35
        min_size = np.random.randint(5, 50)
        max_size = np.random.randint(50, 500)
        if min_size > max_size:
            min_size, max_size = max_size, min_size

        groups = generate_group_samples(n_groups,
                                        min_size=min_size,
                                        max_size=max_size,
                                        inject_extrema=inject_extrema)

        vertical = bool(np.random.rand() < 0.6)
        showfliers = bool(np.random.rand() < 0.85)
        notch = bool(np.random.rand() < 0.2)
        showmeans = bool(np.random.rand() < 0.3)
        whis_options = ['range', 1.5, 2.0, 3.0]
        whis = np.random.choice(whis_options)
        widths = float(np.random.uniform(0.4, 0.9))

        xlabel = 'Category'
        ylabel = 'Value'
        title = f'Random Boxplot {i+1:04d}'

        filename = os.path.join(output_dir, f'boxplot_{i+1:04d}.png')

        meta = create_and_save_boxplot(groups,
                                       filename,
                                       vertical=vertical,
                                       showfliers=showfliers,
                                       notch=notch,
                                       showmeans=showmeans,
                                       whis=whis,
                                       widths=widths,
                                       xlabel=xlabel,
                                       ylabel=ylabel,
                                       title=title,
                                       use_braille=False,
                                       save_data=True)
        manifest.append(meta)

    # Save manifest
    manifest_path = os.path.join(output_dir, 'manifest.json')
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    print(f"Generated {len(manifest)} examples in {output_dir}")
    return manifest

# Example run (adjust n_examples as needed). Use a small number by default to avoid long runs.
# To produce a larger dataset for ML, set n_examples to 1000 or more.
manifest = generate_dataset(output_dir='boxplot_dataset', n_examples=200, seed=42)

Generated 200 examples in boxplot_dataset


Notes
- Each PNG has a corresponding .npz file containing the raw arrays (one array per group) and a .json metadata file describing the plot parameters.
- To enable Braille labels, call create_and_save_boxplot with use_braille=True. The Braille conversion functions are included earlier.